## Import libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

## Load files

In [ ]:


DATA_DIR = Path(".")   # change to your folder path if needed

xml_path = "xml_output.csv"
transactions_path = "transactions.csv"
accounts_path = "accounts.csv"
graph_edges_path = "graph_edges.csv"

xml = pd.read_csv(xml_path, dtype=str)
tx = pd.read_csv(transactions_path, dtype=str)
accounts = pd.read_csv(accounts_path, dtype=str)
graph_edges = pd.read_csv(graph_edges_path, dtype=str)

xml = xml.reset_index().rename(columns={"index": "xml_row_id"})



## Clean strings 
### Clean "1,000 NPR" -> 1000
### Clean "2026-06-14T15:42:08" -> "2026-06-14"
### Remove empty values


In [4]:
def to_number(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace(",", "", regex=False)
         .str.extract(r"([-+]?\d*\.?\d+)", expand=False),
        errors="coerce"
    )


def clean_date(s):
    return pd.to_datetime(s, errors="coerce").dt.date.astype("string")


def first_non_null(series):
    s = series.dropna().astype(str)
    s = s[(s != "") & (s.str.lower() != "nan")]
    return s.iloc[0] if len(s) else pd.NA


def unique_join(series):
    vals = []

    for x in series.dropna().astype(str):
        x = x.strip()

        if x and x.lower() != "nan" and x not in vals:
            vals.append(x)

    return ", ".join(vals) if vals else pd.NA



## Cleaning transaction columns
#### one row contained two transactions, now we have two rows, for same account, having two transactions to make ure all transactions are properly tracked. Cross border is also tracked for each transactions.

In [6]:

def wide_xml_transactions_to_long(xml_df, max_txns=6):
    rows = []

    for i in range(1, max_txns + 1):
        prefix = "report_transaction_" if i == 1 else f"report_transaction_{i}_transaction_"

        colmap = {
            "xml_transaction_number": prefix + "transactionnumber",
            "xml_internal_ref_number": prefix + "internal_ref_number",
            "xml_transaction_datetime": prefix + "date_transaction",
            "xml_transaction_type": prefix + "transmode_comment",
            "xml_transaction_amount_local": prefix + "amount_local",

            "xml_from_institution": prefix + "t_from_my_client_from_account_institution_name",
            "xml_from_account_number": prefix + "t_from_my_client_from_account_account",
            "xml_from_account_name": prefix + "t_from_my_client_from_account_account_name",
            "xml_from_country": prefix + "t_from_my_client_from_country",

            "xml_to_institution": prefix + "t_to_to_account_institution_name",
            "xml_to_account_number": prefix + "t_to_to_account_account",
            "xml_to_account_name": prefix + "t_to_to_account_account_name",
            "xml_to_country": prefix + "t_to_to_country",
        }

        existing = {new: old for new, old in colmap.items() if old in xml_df.columns}

        if not existing:
            continue

        base_cols = [
            "xml_row_id",
            "report_report_id",
            "report_entity_reference",
            "report_submission_date",
            "report_reason",
            "report_report_indicators_indicator"
        ]

        base_cols = [c for c in base_cols if c in xml_df.columns]

        part = xml_df[base_cols + list(existing.values())].copy()
        part = part.rename(columns={v: k for k, v in existing.items()})
        part["xml_transaction_position"] = i

        rows.append(part)

    long_df = pd.concat(rows, ignore_index=True)

    required_any = [
        c for c in [
            "xml_transaction_number",
            "xml_transaction_datetime",
            "xml_transaction_amount_local"
        ]
        if c in long_df.columns
    ]

    long_df = long_df.dropna(subset=required_any, how="all")

    long_df["xml_transaction_date"] = clean_date(long_df["xml_transaction_datetime"])
    long_df["xml_amount_num"] = to_number(long_df["xml_transaction_amount_local"])

    long_df["xml_cross_border_flag"] = (
        long_df["xml_from_country"].astype(str).str.upper()
        != long_df["xml_to_country"].astype(str).str.upper()
    ).astype(int)

    return long_df


xml_txn_long = wide_xml_transactions_to_long(xml)


## Merging accounts.csv

In [7]:

account_cols = [
    "account_id",
    "account_number",
    "institution",
    "branch",
    "acct_type",
    "risk_grade",
    "is_person",
    "name",
    "pep_flag",
    "sanctions_hit",
    "city",
    "opened"
]

account_cols = [c for c in account_cols if c in accounts.columns]

accounts_small = accounts[account_cols].drop_duplicates("account_number").copy()

from_acct = accounts_small.add_prefix("from_acct_")
to_acct = accounts_small.add_prefix("to_acct_")

xml_txn_long = xml_txn_long.merge(
    from_acct,
    how="left",
    left_on="xml_from_account_number",
    right_on="from_acct_account_number"
)

xml_txn_long = xml_txn_long.merge(
    to_acct,
    how="left",
    left_on="xml_to_account_number",
    right_on="to_acct_account_number"
)

## Merging transactionos.csv

In [9]:


tx = tx.copy()

tx["tx_date_clean"] = clean_date(tx["Date"])
tx["tx_amount_num"] = to_number(tx["amount_local_npr"])

tx_cols_needed = [
    "row_index",
    "Date",
    "Time",
    "Sender_account",
    "Receiver_account",
    "sender_account_number",
    "receiver_account_number",
    "Payment_type",
    "transmode_code",
    "amount_local_npr",
    "cross_border_flag",
    "currency_mismatch",

    "sender_institution",
    "receiver_institution",
    "sender_risk_grade",
    "sender_account_type",
    "sender_opened",
    "sender_pep",
    "sender_sanctions",
    "sender_account_age_days",

    "receiver_pep",
    "receiver_sanctions",
    "receiver_account_age_days",

    "velocity_sum_10tx",
    "tx_count_10",
    "tx_count_30",
    "amount_zscore",
    "above_1M_NPR",
    "above_10M_NPR",

    "tx_date_clean",
    "tx_amount_num"
]

tx_cols_needed = [c for c in tx_cols_needed if c in tx.columns]

tx_small = tx[tx_cols_needed].copy()

merged_txn = xml_txn_long.merge(
    tx_small,
    how="left",
    left_on=[
        "xml_from_account_number",
        "xml_to_account_number",
        "xml_transaction_date"
    ],
    right_on=[
        "sender_account_number",
        "receiver_account_number",
        "tx_date_clean"
    ],
    suffixes=("", "_txcsv")
)

merged_txn["amount_abs_diff"] = (
    merged_txn["xml_amount_num"] - merged_txn["tx_amount_num"]
).abs()

merged_txn["tx_amount_match"] = merged_txn["amount_abs_diff"].le(1.0)

merged_txn = merged_txn.sort_values(
    ["xml_row_id", "xml_transaction_position", "amount_abs_diff"],
    na_position="last"
)

merged_txn = merged_txn.drop_duplicates(
    ["xml_row_id", "xml_transaction_position"],
    keep="first"
)

## Aggregate back to one row per STR report

In [10]:

report_df = merged_txn.groupby("xml_row_id", as_index=False).agg(
    report_id=("report_report_id", first_non_null),
    entity_reference=("report_entity_reference", first_non_null),
    narrative=("report_reason", first_non_null),

    # Direct structured XML fields
    institution_name_xml=("xml_from_institution", first_non_null),
    customer_name_xml=("xml_from_account_name", first_non_null),
    start_date_xml=("xml_transaction_date", "min"),
    end_date_xml=("xml_transaction_date", "max"),
    transaction_count_xml=("xml_transaction_number", "count"),
    transaction_type_xml=("xml_transaction_type", unique_join),
    aggregate_amount_xml=("xml_amount_num", "sum"),
    cross_border_count_xml=("xml_cross_border_flag", "sum"),
    counterparty_names_xml=("xml_to_account_name", unique_join),
    typology_xml=("report_report_indicators_indicator", unique_join),

    # From accounts.csv
    customer_account_number=("xml_from_account_number", first_non_null),
    customer_risk_grade_accounts=("from_acct_risk_grade", first_non_null),
    customer_account_type_accounts=("from_acct_acct_type", first_non_null),
    customer_is_person_accounts=("from_acct_is_person", first_non_null),
    customer_city_accounts=("from_acct_city", first_non_null),
    customer_opened_accounts=("from_acct_opened", first_non_null),
    customer_pep_accounts=("from_acct_pep_flag", first_non_null),
    customer_sanctions_accounts=("from_acct_sanctions_hit", first_non_null),

    counterparty_account_numbers=("xml_to_account_number", unique_join),
    counterparty_names_accounts=("to_acct_name", unique_join),
    counterparty_sanctions_accounts=("to_acct_sanctions_hit", unique_join),

    # From transactions.csv
    txcsv_rows_matched=("row_index", lambda s: s.notna().sum()),
    payment_type_txcsv=("Payment_type", unique_join),
    aggregate_amount_txcsv=("tx_amount_num", "sum"),
    cross_border_count_txcsv=("cross_border_flag", lambda s: pd.to_numeric(s, errors="coerce").fillna(0).sum()),

    sender_risk_grade_txcsv=("sender_risk_grade", first_non_null),
    sender_account_age_days_txcsv=("sender_account_age_days", first_non_null),
    sender_sanctions_txcsv=("sender_sanctions", first_non_null),
    receiver_sanctions_txcsv=("receiver_sanctions", unique_join),

    velocity_sum_10tx_txcsv=("velocity_sum_10tx", first_non_null),
    tx_count_10_txcsv=("tx_count_10", first_non_null),
    tx_count_30_txcsv=("tx_count_30", first_non_null),
    amount_zscore_txcsv=("amount_zscore", first_non_null),
    above_1M_NPR_txcsv=("above_1M_NPR", first_non_null),
    above_10M_NPR_txcsv=("above_10M_NPR", first_non_null),
)

## Add empty column to dataframe, to be obtained from the narrative

In [ ]:


narrative_only_fields = [
    "alert_trigger",
    "kyc_profile_claim",
    "profile_mismatch_claim",
    "usual_corridor_claim",
    "threshold_avoidance_claim",
    "velocity_claim",
    "relationship_age_claim",
    "adverse_media_claim",
    "sanctions_claim",
    "reporting_officer_judgement",
    "reason_for_escalation",
    "risk_disclaimer"
]

for col in narrative_only_fields:
    report_df[col] = pd.NA


## Final combined dataframe

In [ ]:


df = report_df.copy()

print("Combined dataframe shape:", df.shape)
display(df.head())


Combined dataframe shape: (276, 51)


,xml_row_id,report_id,entity_reference,narrative,institution_name_xml,customer_name_xml,start_date_xml,end_date_xml,transaction_count_xml,transaction_type_xml,...,profile_mismatch_claim,usual_corridor_claim,threshold_avoidance_claim,velocity_claim,relationship_age_claim,adverse_media_claim,sanctions_claim,reporting_officer_judgement,reason_for_escalation,risk_disclaimer
0,0,RPT-2026-000001,NIMB-2026-000001,Suspicious transaction observed.,PCBL,John Jensen,2022-10-07,2022-10-07,1,Cash Deposit transfer,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,1,RPT-2026-000002,SBL-2026-000002,During the branch's periodic transaction-monit...,ADBL,Sally Dominguez,2022-10-07,2022-10-07,1,Cash Withdrawal transfer,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2,RPT-2026-000003,RBBL-2026-000003,During the branch's periodic transaction-monit...,KUMARI,"Cook, Flores and Ray",2022-10-07,2022-10-07,1,Cross-border transfer,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,3,RPT-2026-000004,HBL-2026-000004,During the branch's periodic transaction-monit...,PRABHU,Robert King,2022-10-07,2022-10-07,1,Cross-border transfer,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,4,RPT-2026-000005,MBL-2026-000005,Suspicious transaction observed.,MEGA,Porter LLC,2022-10-07,2022-10-07,1,Debit card transfer,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## individual column values

In [15]:
row = df.iloc[1, :]
print("\nNarrative for df.iloc[1, :]:")
print(row["narrative"])

print("\nFull row values for df.iloc[1, :]:")
display(row.to_frame(name="value"))


Narrative for df.iloc[1, :]:
During the branch's periodic transaction-monitoring review, the compliance desk at ADBL examined the account held by Sally Dominguez after an internal threshold alert was triggered. Between 2022-10-07 and 2022-10-07, the customer conducted 1 transaction(s) — predominantly cash withdrawals — amounting to approximately NPR 21,807. Of these, 1 were cross-border, routed to counterparties outside the customer's usual corridor. The principal counterparties observed were Henry King. For context, the customer's KYC file lists a modest declared income profile; in the reviewing officer's judgement the velocity and aggregate value of the funds are not fully consistent with that profile. It is worth noting that no adverse media or sanctions matches were identified at the time of filing, and the customer has been with the institution for some years, which the desk weighed before escalating. Nonetheless, a number of the transfers appear to have been deliberately kept be

,value
xml_row_id,1
report_id,RPT-2026-000002
entity_reference,SBL-2026-000002
narrative,During the branch's periodic transaction-monit...
institution_name_xml,ADBL
customer_name_xml,Sally Dominguez
start_date_xml,2022-10-07
end_date_xml,2022-10-07
transaction_count_xml,1
transaction_type_xml,Cash Withdrawal transfer


## Narrative for the row

In [19]:
import textwrap

row = df.iloc[1, :]

print("Narrative for df.iloc[1, :]\n")
print(textwrap.fill(row["narrative"], width=100))

Narrative for df.iloc[1, :]

During the branch's periodic transaction-monitoring review, the compliance desk at ADBL examined the
account held by Sally Dominguez after an internal threshold alert was triggered. Between 2022-10-07
and 2022-10-07, the customer conducted 1 transaction(s) — predominantly cash withdrawals — amounting
to approximately NPR 21,807. Of these, 1 were cross-border, routed to counterparties outside the
customer's usual corridor. The principal counterparties observed were Henry King. For context, the
customer's KYC file lists a modest declared income profile; in the reviewing officer's judgement the
velocity and aggregate value of the funds are not fully consistent with that profile. It is worth
noting that no adverse media or sanctions matches were identified at the time of filing, and the
customer has been with the institution for some years, which the desk weighed before escalating.
Nonetheless, a number of the transfers appear to have been deliberately kept bel

## fields extraction from the narrative

In [23]:
import re
import pandas as pd
import numpy as np
from IPython.display import display


def clean_text(text):
    if pd.isna(text):
        return ""
    return str(text).strip()


def extract_first(pattern, text, flags=re.IGNORECASE):
    match = re.search(pattern, text, flags)
    if match:
        return match.group(1).strip()
    return pd.NA


def extract_amount(text):
    """
    Extracts amount like:
    NPR 21,807
    NPR 1,50,000
    approximately NPR 21,807
    """
    match = re.search(
        r"(?:amounting to approximately|amounting to|aggregate value of|approximately)?\s*(NPR\s*[\d,]+(?:\.\d+)?)",
        text,
        re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return pd.NA


def normalize_amount(amount_text):
    if pd.isna(amount_text):
        return pd.NA

    s = str(amount_text)
    s = s.replace("NPR", "")
    s = s.replace(",", "")
    s = s.strip()

    try:
        return float(s)
    except:
        return pd.NA


def extract_dates(text):
    dates = re.findall(r"\d{4}-\d{2}-\d{2}", text)

    if len(dates) >= 2:
        return dates[0], dates[1]
    elif len(dates) == 1:
        return dates[0], dates[0]
    else:
        return pd.NA, pd.NA


def yes_no_claim(text, patterns):
    text_lower = text.lower()

    for pattern in patterns:
        if re.search(pattern, text_lower, re.IGNORECASE):
            return 1

    return 0


# -----------------------------
# Main narrative extractor
# -----------------------------

def extract_fields_from_narrative(narrative):
    text = clean_text(narrative)

    start_date, end_date = extract_dates(text)

    institution_name = extract_first(
        r"compliance desk at ([A-Za-z0-9&.\- ]+?) examined",
        text
    )

    customer_name = extract_first(
    r"account held by\s+(.+?)\s+after\s+an?\s+",
    text
    )

    alert_trigger = extract_first(
        r"after an? (.+? alert) was triggered",
        text
    )

    transaction_count = extract_first(
        r"customer conducted\s+(\d+)\s+transaction",
        text
    )

    transaction_type = extract_first(
        r"predominantly\s+(.+?)\s+—\s+amounting",
        text
    )

    aggregate_amount = extract_amount(text)
    aggregate_amount_num = normalize_amount(aggregate_amount)

    cross_border_count = extract_first(
        r"Of these,\s+(\d+)\s+were cross-border",
        text
    )

    counterparty_names = extract_first(
        r"principal counterparties observed were\s+(.+?)\.",
        text
    )

    typology = extract_first(
        r"consistent with\s+(.+?)\.",
        text
    )

    # -----------------------------
    # Soft analytical claims
    # -----------------------------

    kyc_profile_claim = extract_first(
        r"KYC file lists\s+(.+?);",
        text
    )

    profile_mismatch_claim = extract_first(
        r"judgement\s+(.+?profile)",
        text
    )

    usual_corridor_claim = extract_first(
        r"(outside the customer's usual corridor)",
        text
    )

    threshold_avoidance_claim = extract_first(
        r"(kept below reporting thresholds)",
        text
    )

    velocity_claim = extract_first(
        r"(executed in rapid succession)",
        text
    )

    relationship_age_claim = extract_first(
        r"(customer has been with the institution for some years)",
        text
    )

    adverse_media_claim = extract_first(
        r"(no adverse media[^,.]+)",
        text
    )

    sanctions_claim = extract_first(
        r"(no sanctions matches[^,.]+)",
        text
    )

    reporting_officer_judgement = extract_first(
        r"in the reviewing officer's judgement\s+(.+?)\.",
        text
    )

    reason_for_escalation = extract_first(
        r"before escalating\. Nonetheless,\s+(.+?)\.",
        text
    )

    risk_disclaimer = extract_first(
        r"(does not, by itself, constitute a determination of wrongdoing)",
        text
    )

    return {
        "institution_name": institution_name,
        "customer_name": customer_name,
        "alert_trigger": alert_trigger,

        "start_date": start_date,
        "end_date": end_date,
        "transaction_count": transaction_count,
        "transaction_type": transaction_type,
        "aggregate_amount": aggregate_amount,
        "aggregate_amount_num": aggregate_amount_num,
        "cross_border_count": cross_border_count,
        "counterparty_names": counterparty_names,
        "typology": typology,

        "kyc_profile_claim": kyc_profile_claim,
        "profile_mismatch_claim": profile_mismatch_claim,
        "usual_corridor_claim": usual_corridor_claim,
        "threshold_avoidance_claim": threshold_avoidance_claim,
        "velocity_claim": velocity_claim,
        "relationship_age_claim": relationship_age_claim,

        "adverse_media_claim": adverse_media_claim,
        "sanctions_claim": sanctions_claim,

        "reporting_officer_judgement": reporting_officer_judgement,
        "reason_for_escalation": reason_for_escalation,
        "risk_disclaimer": risk_disclaimer,
    }

## apply extraction to every row

In [24]:
extracted_df = df["narrative"].apply(extract_fields_from_narrative).apply(pd.Series)

# Add suffix so we know these came from narrative
extracted_df = extracted_df.add_suffix("_from_narrative")

# Combine with original df
df_extracted = pd.concat([df, extracted_df], axis=1)

print("Original df shape:", df.shape)
print("After narrative extraction:", df_extracted.shape)

display(df_extracted.head())

Original df shape: (276, 51)
After narrative extraction: (276, 74)


,xml_row_id,report_id,entity_reference,narrative,institution_name_xml,customer_name_xml,start_date_xml,end_date_xml,transaction_count_xml,transaction_type_xml,...,profile_mismatch_claim_from_narrative,usual_corridor_claim_from_narrative,threshold_avoidance_claim_from_narrative,velocity_claim_from_narrative,relationship_age_claim_from_narrative,adverse_media_claim_from_narrative,sanctions_claim_from_narrative,reporting_officer_judgement_from_narrative,reason_for_escalation_from_narrative,risk_disclaimer_from_narrative
0,0,RPT-2026-000001,NIMB-2026-000001,Suspicious transaction observed.,PCBL,John Jensen,2022-10-07,2022-10-07,1,Cash Deposit transfer,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN
1,1,RPT-2026-000002,SBL-2026-000002,During the branch's periodic transaction-monit...,ADBL,Sally Dominguez,2022-10-07,2022-10-07,1,Cash Withdrawal transfer,...,the velocity and aggregate value of the funds ...,outside the customer's usual corridor,kept below reporting thresholds,executed in rapid succession,customer has been with the institution for som...,no adverse media or sanctions matches were ide...,<NA>,the velocity and aggregate value of the funds ...,a number of the transfers appear to have been ...,"does not, by itself, constitute a determinatio..."
2,2,RPT-2026-000003,RBBL-2026-000003,During the branch's periodic transaction-monit...,KUMARI,"Cook, Flores and Ray",2022-10-07,2022-10-07,1,Cross-border transfer,...,the velocity and aggregate value of the funds ...,outside the customer's usual corridor,kept below reporting thresholds,executed in rapid succession,customer has been with the institution for som...,no adverse media or sanctions matches were ide...,<NA>,the velocity and aggregate value of the funds ...,a number of the transfers appear to have been ...,"does not, by itself, constitute a determinatio..."
3,3,RPT-2026-000004,HBL-2026-000004,During the branch's periodic transaction-monit...,PRABHU,Robert King,2022-10-07,2022-10-07,1,Cross-border transfer,...,the velocity and aggregate value of the funds ...,outside the customer's usual corridor,kept below reporting thresholds,executed in rapid succession,customer has been with the institution for som...,no adverse media or sanctions matches were ide...,<NA>,the velocity and aggregate value of the funds ...,a number of the transfers appear to have been ...,"does not, by itself, constitute a determinatio..."
4,4,RPT-2026-000005,MBL-2026-000005,Suspicious transaction observed.,MEGA,Porter LLC,2022-10-07,2022-10-07,1,Debit card transfer,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN


In [25]:
row = df_extracted.iloc[1, :]

print("Narrative:\n")
print(row["narrative"])

print("\nExtracted fields from narrative:\n")

cols_to_check = [col for col in df_extracted.columns if col.endswith("_from_narrative")]

display(row[cols_to_check].to_frame(name="extracted_value"))

Narrative:

During the branch's periodic transaction-monitoring review, the compliance desk at ADBL examined the account held by Sally Dominguez after an internal threshold alert was triggered. Between 2022-10-07 and 2022-10-07, the customer conducted 1 transaction(s) — predominantly cash withdrawals — amounting to approximately NPR 21,807. Of these, 1 were cross-border, routed to counterparties outside the customer's usual corridor. The principal counterparties observed were Henry King. For context, the customer's KYC file lists a modest declared income profile; in the reviewing officer's judgement the velocity and aggregate value of the funds are not fully consistent with that profile. It is worth noting that no adverse media or sanctions matches were identified at the time of filing, and the customer has been with the institution for some years, which the desk weighed before escalating. Nonetheless, a number of the transfers appear to have been deliberately kept below reporting thre

,extracted_value
institution_name_from_narrative,ADBL
customer_name_from_narrative,Sally Dominguez
alert_trigger_from_narrative,internal threshold alert
start_date_from_narrative,2022-10-07
end_date_from_narrative,2022-10-07
transaction_count_from_narrative,1
transaction_type_from_narrative,cash withdrawals
aggregate_amount_from_narrative,"NPR 21,807"
aggregate_amount_num_from_narrative,21807.0
cross_border_count_from_narrative,1


In [26]:
from IPython.display import display
import pandas as pd

# choose the row you want to inspect
row_idx = 1

# use df if you only want the combined dataframe
# use df_extracted if you already concatenated narrative-extracted columns
row = df.iloc[row_idx, :]

actual_field_map = {
    "institution_name": "institution_name_xml",
    "customer_name": "customer_name_xml",
    "alert_trigger": "alert_trigger",

    "start_date": "start_date_xml",
    "end_date": "end_date_xml",
    "transaction_count": "transaction_count_xml",
    "transaction_type": "transaction_type_xml",
    "aggregate_amount": "aggregate_amount_xml",
    "cross_border_count": "cross_border_count_xml",
    "counterparty_names": "counterparty_names_xml",
    "typology": "typology_xml",

    "kyc_profile_claim": "kyc_profile_claim",
    "profile_mismatch_claim": "profile_mismatch_claim",
    "usual_corridor_claim": "usual_corridor_claim",
    "threshold_avoidance_claim": "threshold_avoidance_claim",
    "velocity_claim": "velocity_claim",
    "relationship_age_claim": "relationship_age_claim",

    "adverse_media_claim": "adverse_media_claim",
    "sanctions_claim": "sanctions_claim",

    "reporting_officer_judgement": "reporting_officer_judgement",
    "reason_for_escalation": "reason_for_escalation",
    "risk_disclaimer": "risk_disclaimer",
}

actual_values = []

for field, col in actual_field_map.items():
    value = row[col] if col in df.columns else pd.NA

    actual_values.append({
        "field": field,
        "dataframe_column": col,
        "actual_value": value
    })

actual_values_df = pd.DataFrame(actual_values)

print("Narrative for this row:\n")
print(row["narrative"])

print("\nActual values from dataframe:\n")
display(actual_values_df)

Narrative for this row:

During the branch's periodic transaction-monitoring review, the compliance desk at ADBL examined the account held by Sally Dominguez after an internal threshold alert was triggered. Between 2022-10-07 and 2022-10-07, the customer conducted 1 transaction(s) — predominantly cash withdrawals — amounting to approximately NPR 21,807. Of these, 1 were cross-border, routed to counterparties outside the customer's usual corridor. The principal counterparties observed were Henry King. For context, the customer's KYC file lists a modest declared income profile; in the reviewing officer's judgement the velocity and aggregate value of the funds are not fully consistent with that profile. It is worth noting that no adverse media or sanctions matches were identified at the time of filing, and the customer has been with the institution for some years, which the desk weighed before escalating. Nonetheless, a number of the transfers appear to have been deliberately kept below r

,field,dataframe_column,actual_value
0,institution_name,institution_name_xml,ADBL
1,customer_name,customer_name_xml,Sally Dominguez
2,alert_trigger,alert_trigger,<NA>
3,start_date,start_date_xml,2022-10-07
4,end_date,end_date_xml,2022-10-07
5,transaction_count,transaction_count_xml,1
6,transaction_type,transaction_type_xml,Cash Withdrawal transfer
7,aggregate_amount,aggregate_amount_xml,21807.13
8,cross_border_count,cross_border_count_xml,1
9,counterparty_names,counterparty_names_xml,Henry King


In [27]:
row_actual = df.iloc[row_idx, :]
row_extracted = df_extracted.iloc[row_idx, :]

comparison = []

for field, actual_col in actual_field_map.items():
    extracted_col = field + "_from_narrative"

    actual_value = row_actual[actual_col] if actual_col in df.columns else pd.NA
    extracted_value = row_extracted[extracted_col] if extracted_col in df_extracted.columns else pd.NA

    comparison.append({
        "field": field,
        "actual_dataframe_column": actual_col,
        "actual_value": actual_value,
        "narrative_extracted_column": extracted_col,
        "narrative_extracted_value": extracted_value
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

,field,actual_dataframe_column,actual_value,narrative_extracted_column,narrative_extracted_value
0,institution_name,institution_name_xml,ADBL,institution_name_from_narrative,ADBL
1,customer_name,customer_name_xml,Sally Dominguez,customer_name_from_narrative,Sally Dominguez
2,alert_trigger,alert_trigger,<NA>,alert_trigger_from_narrative,internal threshold alert
3,start_date,start_date_xml,2022-10-07,start_date_from_narrative,2022-10-07
4,end_date,end_date_xml,2022-10-07,end_date_from_narrative,2022-10-07
5,transaction_count,transaction_count_xml,1,transaction_count_from_narrative,1
6,transaction_type,transaction_type_xml,Cash Withdrawal transfer,transaction_type_from_narrative,cash withdrawals
7,aggregate_amount,aggregate_amount_xml,21807.13,aggregate_amount_from_narrative,"NPR 21,807"
8,cross_border_count,cross_border_count_xml,1,cross_border_count_from_narrative,1
9,counterparty_names,counterparty_names_xml,Henry King,counterparty_names_from_narrative,Henry King
